# 🧠💥🧠 Notebook 1: Split Brain — Two Leaders at Once

**The setup:** Node A holds a lease and is the leader. The network partitions A from the rest of the cluster. The cluster decides A is dead and elects B. The partition heals — and *both* think they're in charge.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/split-brain-and-fencing
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 Simulate the partition

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Storage:
    log: List[str] = field(default_factory=list)
    def write(self, who, value):
        self.log.append(f'{who}: {value}')

store = Storage()

# A is leader, takes a long GC pause (~partition).
leader = 'A'
store.write(leader, 'config v1')

# Cluster can't see A, elects B as the new leader.
leader = 'B'
store.write(leader, 'config v2')

# A wakes up still believing it's the leader and writes again.
store.write('A', 'config v1.1 (STALE)')
store.write('B', 'config v3')

for entry in store.log:
    print(' ', entry)


The storage layer happily accepted a write from a *stale* leader. In production this is how you get corrupted databases, double-charged customers, and lost data.

👉 Next notebook: **fencing tokens** — a monotonically increasing number that lets storage reject stale writers.